# Train CIFAR10 → integer-quantized FC stack (parametric)

A **parametric** trainer that targets the same DiNN-style integer-weight format the C++ FHE inference pipeline already reads (see `MNIST_30/main.cpp`, `MNIST_100/main.cpp`, `cifar10/main.cpp`, and `src/io/csv.cpp`).

Pick the network shape and hidden activation in **CONFIG** (see § 2). Everything else — preprocessing, training, plotting, headroom checks, CSV export, exact round-trip verification — is identical to the worked example in `train_cifar100_baseline.ipynb`.

**What CONFIG controls**

* `topology` — any FC stack starting at 3072 (32·32·3 RGB, HWC-flat) and ending at 10 (10 classes).
* `activation` — hidden-layer LUT activation (one of `sign`, `heaviside`, `ternary_act`, `staircase`, `hardtanh_q`, `sigmoid_q`, `relu_q`, `square_q`). The trainer's STE forward matches the C++ LUT bit-for-bit *as long as the pre-activation `|s|` stays under `preShift`* (see § 11).
* `layer_quant` — per-layer integer quantization (`ternary` for `{-1, 0, +1}` or `int` for signed integers in `[-max_int, +max_int]`). Biases are always rounded to integer.
* `prefix`, `output_dir` — control where the CSVs are written and what the C++ side expects to load.

The notebook follows the same section layout as `train_cifar100_baseline.ipynb` (imports → CONFIG → loader → preprocessing → activation factory → MLP → train → plots → eval → headroom → export → reload check → summary).

## 1. Setup and imports

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

## 2. Configuration

Everything tunable lives in `CONFIG`. The default reproduces a DiNN-30 (`784 → 30 → 10`) sign-activated MLP — i.e. the `MNIST_30/` sub-project's expected weights. See the **CONFIG cookbook** in `new_models/README.md` for ready-to-paste alternatives (DiNN-100, stacked, etc.).

In [ ]:
CONFIG = {
    # ── Training ─────────────────────────────────────────────────────────
    "epochs": 30, "batch_size": 256, "lr": 1e-3, "weight_decay": 0.0, "seed": 0,
    "device": "cuda" if torch.cuda.is_available() else "cpu",

    # ── I/O ──────────────────────────────────────────────────────────────
    "data_dir":   "./data",
    "output_dir": "cifar10",      # CSVs land in new_models/<output_dir>/
    "prefix":     "cifar10_weights",  # CSV file prefix, matches the C++ sub-project

    # ── Architecture ─────────────────────────────────────────────────────
    # topology must start at 3072 (32*32*3 RGB, HWC-flat) and end at 10 (10 classes).
    # Add as many hidden widths in between as you want.
    "topology":   [3072, 30, 10],

    # Hidden activation, applied after every Linear *except* the last.
    # Choices: sign | heaviside | ternary_act | staircase | hardtanh_q
    #        | sigmoid_q | relu_q | square_q
    "activation":   "sign",
    "act_params":   {},   # passed to make_activation; see § 6 for keys per variant

    # ── Per-Linear integer quantization (one entry per Linear layer) ─────
    # kind="ternary": W -> sign(W) * 1[|W| >= thresh*mean(|W|)] in {-1, 0, +1}
    # kind="int":     W -> round(W * (max_int / max|W|)) clipped to [-max_int, +max_int]
    "layer_quant": [
        {"kind": "ternary", "thresh": 0.7},   # fc1 (W1)
        {"kind": "int",     "max_int": 15},   # fc2 (W2)
    ],

    # ── FHE-side headroom defaults (used only by § 11 sanity checks) ─────
    # The trainer never sees `preShift`; these mirror the values the C++
    # main.cpp uses so the headroom warnings match deployment.
    "preShift":   256,    # hidden-layer LUT pre-shift; |s_hidden| must stay below this
    "pOutput":    1024,   # modular decryption period for output; |s_output| must be < pOutput/2
}


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything(CONFIG["seed"])
DEVICE = torch.device(CONFIG["device"])

assert CONFIG["topology"][0]  == 3072, "CIFAR10 input must be 3072 (32*32*3 RGB)."
assert CONFIG["topology"][-1] == 10,   "CIFAR10 output must be 10 classes."
assert len(CONFIG["layer_quant"]) == len(CONFIG["topology"]) - 1, \
    "layer_quant must have one entry per Linear layer."

print("Device:    ", DEVICE)
print("Output dir:", Path(CONFIG["output_dir"]).resolve())
print("Topology:  ", " -> ".join(map(str, CONFIG["topology"])),
      f"  ({len(CONFIG['topology']) - 1} Linear layer(s))")
print("Activation:", CONFIG["activation"], CONFIG["act_params"])
print("Quant per layer:", CONFIG["layer_quant"])

## 3. CIFAR10 dataset loading

We download CIFAR10 with no transform so we keep the raw PIL images and can preprocess them ourselves — the same way `io::LoadImageBipolar(path, 3072, /*channels=*/3)` does on the C++ side.


In [ ]:
train_pil = datasets.CIFAR10(
    root=CONFIG["data_dir"], train=True,  download=True, transform=None
)
test_pil = datasets.CIFAR10(
    root=CONFIG["data_dir"], train=False, download=True, transform=None
)

print("Train:", len(train_pil), " Test:", len(test_pil))
print("Number of classes:", len(train_pil.classes))

_img, _label = train_pil[0]
print("Sample:", _img.size, _img.mode, "label =", _label)

## 4. C++-compatible preprocessing

The C++ inference path does:

```cpp
// stb_image returns flat HWC interleaved RGB bytes when called with channels=3
unsigned char* data = stbi_load(path, &w, &h, &ch, /*channels=*/3);
for (int i = 0; i < 32*32*3; ++i) out[i] = (data[i] > 127) ? +1 : -1;
```

We replicate that exactly:

1. Convert PIL → NumPy `uint8` array of shape `(H, W, 3)` in **HWC** order.
2. Threshold each raw pixel byte at 127.
3. Map `> 127 → +1.0`, otherwise `-1.0`.
4. Flatten in HWC order (row, then column, then channel) to length 3072.

We deliberately **do not** use PyTorch's default `ToTensor` / CHW conversion — that would put all R bytes first, then all G, then all B, which is **not** the layout the C++ inference loop expects.


In [ ]:
def pil_to_bipolar_hwc(img) -> np.ndarray:
    """PIL → NumPy HWC uint8 → threshold at 127 → {-1, +1} float32, flat length 3072."""
    arr = np.asarray(img.convert("RGB"), dtype=np.uint8)  # (H, W, 3) HWC
    assert arr.shape == (32, 32, 3), f"unexpected shape {arr.shape}"
    bipolar = np.where(arr > 127, np.float32(1.0), np.float32(-1.0))
    flat = bipolar.reshape(-1)  # HWC row-major: (h, w, c) → single 3072-vector
    assert flat.shape == (3072,)
    return flat


def encode_dataset(pil_dataset) -> tuple[np.ndarray, np.ndarray]:
    n = len(pil_dataset)
    X = np.empty((n, 3072), dtype=np.float32)
    y = np.empty((n,),     dtype=np.int64)
    for i in range(n):
        img, label = pil_dataset[i]
        X[i] = pil_to_bipolar_hwc(img)
        y[i] = label
    return X, y


X_train, y_train = encode_dataset(train_pil)
X_test,  y_test  = encode_dataset(test_pil)

print("X_train:", X_train.shape, X_train.dtype, "min/max =", X_train.min(), X_train.max())
print("X_test :", X_test.shape,  X_test.dtype,  "min/max =", X_test.min(),  X_test.max())
print("y_train classes:", len(np.unique(y_train)),
      "  y_test classes:", len(np.unique(y_test)))

# Sanity: every value is exactly +1 or -1.
assert set(np.unique(X_train)).issubset({-1.0, 1.0})
assert set(np.unique(X_test)).issubset({-1.0, 1.0})

train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
test_ds  = TensorDataset(torch.from_numpy(X_test),  torch.from_numpy(y_test))
train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True,  num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=CONFIG["batch_size"], shuffle=False, num_workers=0)
print("train batches:", len(train_loader), " test batches:", len(test_loader))


## 5. Hidden-activation factory (LUT + STE)

Each variant follows the same shape as the baseline `SignSTE`: the **forward** pass evaluates a deterministic look-up table on the integer pre-activation `s = W @ x + b`, and the **backward** pass passes the gradient through with a clipped-identity mask (zero gradient where `|s|` exceeds the saturation point) — the standard Bengio/BinaryNet straight-through estimator trick.

The C++ deployed LUT actually evaluates `f(s + preShift)` with the same `f`; as long as `|s| < preShift` (see § 11), the trainer's STE forward and the deployed LUT compute *exactly* the same value.

| `activation`    | Forward (on integer `s`)                                | Output range            | `act_params`                  |
|-----------------|---------------------------------------------------------|-------------------------|-------------------------------|
| `sign`          | `+1 if s >= 0 else -1`                                  | `{-1, +1}`              | —                             |
| `heaviside`     | `1 if s >= 0 else 0`                                    | `{0, 1}`                | —                             |
| `ternary_act`   | `+1 if s > thresh, -1 if s < -thresh, else 0`           | `{-1, 0, +1}`           | `thresh` (default `1.0`)      |
| `staircase`     | `clamp(round(s / scale), -K, K)`                        | integers in `[-K, K]`   | `scale` (1.0), `K` (7)        |
| `hardtanh_q`    | `round(clamp(s, -K, K))`                                | integers in `[-K, K]`   | `K` (7)                       |
| `sigmoid_q`     | `clamp(round(K * sigmoid(s/scale)), 0, K)`              | integers in `[0, K]`    | `scale` (4.0), `K` (7)        |
| `relu_q`        | `clamp(round(s / scale), 0, K)`                         | integers in `[0, K]`    | `scale` (1.0), `K` (7)        |
| `square_q`      | `clamp(round((s/scale)**2), 0, K)`                      | integers in `[0, K]`    | `scale` (8.0), `K` (7)        |

In [ ]:
# ── Straight-through estimators for each LUT activation ────────────────────
#
# Forward = exact LUT (so training-time accuracy == deployed-LUT accuracy
# whenever |s| < preShift). Backward = gradient passed through, masked to
# zero outside the saturation region.

class SignSTE(torch.autograd.Function):
    """forward: +1 if s >= 0 else -1.  backward: clipped identity, |s| <= 1."""

    @staticmethod
    def forward(ctx, s):
        ctx.save_for_backward(s)
        return torch.where(s >= 0, torch.ones_like(s), -torch.ones_like(s))

    @staticmethod
    def backward(ctx, grad_output):
        (s,) = ctx.saved_tensors
        return grad_output * (s.abs() <= 1.0).to(grad_output.dtype)


class HeavisideSTE(torch.autograd.Function):
    """forward: 1 if s >= 0 else 0.  backward: clipped identity, |s| <= 1."""

    @staticmethod
    def forward(ctx, s):
        ctx.save_for_backward(s)
        return torch.where(s >= 0, torch.ones_like(s), torch.zeros_like(s))

    @staticmethod
    def backward(ctx, grad_output):
        (s,) = ctx.saved_tensors
        return grad_output * (s.abs() <= 1.0).to(grad_output.dtype)


class TernaryActSTE(torch.autograd.Function):
    """forward: +1 if s > thr, -1 if s < -thr, else 0.  backward: clipped identity, |s| <= thr+1."""

    @staticmethod
    def forward(ctx, s, thr):
        ctx.save_for_backward(s)
        ctx.thr = float(thr)
        out = torch.zeros_like(s)
        out = torch.where(s >  ctx.thr, torch.ones_like(s),  out)
        out = torch.where(s < -ctx.thr, -torch.ones_like(s), out)
        return out

    @staticmethod
    def backward(ctx, grad_output):
        (s,) = ctx.saved_tensors
        bound = ctx.thr + 1.0
        return grad_output * (s.abs() <= bound).to(grad_output.dtype), None


class StaircaseSTE(torch.autograd.Function):
    """forward: clamp(round(s / scale), -K, K).  backward: identity in |s| <= (K + 0.5) * scale."""

    @staticmethod
    def forward(ctx, s, scale, K):
        ctx.save_for_backward(s)
        ctx.scale = float(scale)
        ctx.K = int(K)
        return torch.round(s / ctx.scale).clamp(-float(ctx.K), float(ctx.K))

    @staticmethod
    def backward(ctx, grad_output):
        (s,) = ctx.saved_tensors
        bound = (ctx.K + 0.5) * ctx.scale
        return grad_output * (s.abs() <= bound).to(grad_output.dtype), None, None


class HardtanhQSTE(torch.autograd.Function):
    """forward: round(clamp(s, -K, K)).  backward: identity in |s| <= K."""

    @staticmethod
    def forward(ctx, s, K):
        ctx.save_for_backward(s)
        ctx.K = float(K)
        return torch.round(s.clamp(-ctx.K, ctx.K))

    @staticmethod
    def backward(ctx, grad_output):
        (s,) = ctx.saved_tensors
        return grad_output * (s.abs() <= ctx.K).to(grad_output.dtype), None


class SigmoidQSTE(torch.autograd.Function):
    """forward: clamp(round(K * sigmoid(s/scale)), 0, K).  backward: identity in |s| <= 4*scale."""

    @staticmethod
    def forward(ctx, s, scale, K):
        ctx.save_for_backward(s)
        ctx.scale = float(scale)
        ctx.K = int(K)
        return torch.round(torch.sigmoid(s / ctx.scale) * ctx.K).clamp(0.0, float(ctx.K))

    @staticmethod
    def backward(ctx, grad_output):
        (s,) = ctx.saved_tensors
        return grad_output * (s.abs() <= 4.0 * ctx.scale).to(grad_output.dtype), None, None


class ReluQSTE(torch.autograd.Function):
    """forward: clamp(round(s / scale), 0, K).  backward: identity in -0.5*scale <= s <= (K+0.5)*scale."""

    @staticmethod
    def forward(ctx, s, scale, K):
        ctx.save_for_backward(s)
        ctx.scale = float(scale)
        ctx.K = int(K)
        return torch.round(s / ctx.scale).clamp(0.0, float(ctx.K))

    @staticmethod
    def backward(ctx, grad_output):
        (s,) = ctx.saved_tensors
        lo = -0.5 * ctx.scale
        hi = (ctx.K + 0.5) * ctx.scale
        mask = (s >= lo) & (s <= hi)
        return grad_output * mask.to(grad_output.dtype), None, None


class SquareQSTE(torch.autograd.Function):
    """forward: clamp(round((s/scale)**2), 0, K).  backward: 2*s/scale * 1[ s^2 <= K*scale^2 ]."""

    @staticmethod
    def forward(ctx, s, scale, K):
        ctx.save_for_backward(s)
        ctx.scale = float(scale)
        ctx.K = int(K)
        return torch.round((s / ctx.scale) ** 2).clamp(0.0, float(ctx.K))

    @staticmethod
    def backward(ctx, grad_output):
        (s,) = ctx.saved_tensors
        bound = (ctx.K + 0.5) ** 0.5 * ctx.scale
        deriv = 2.0 * (s / ctx.scale)
        return grad_output * deriv * (s.abs() <= bound).to(grad_output.dtype), None, None


# ── Activation factory ─────────────────────────────────────────────────────
#
# Returns a fresh nn.Module each call so the MLP can stack one per hidden layer
# (each module is stateless apart from its bound parameters).

def make_activation(name: str, params: dict | None = None) -> nn.Module:
    name = name.lower()
    p = dict(params or {})

    if name == "sign":
        class _Act(nn.Module):
            def forward(self, s): return SignSTE.apply(s)
        return _Act()

    if name == "heaviside":
        class _Act(nn.Module):
            def forward(self, s): return HeavisideSTE.apply(s)
        return _Act()

    if name == "ternary_act":
        thr = float(p.get("thresh", 1.0))
        class _Act(nn.Module):
            def forward(self, s): return TernaryActSTE.apply(s, thr)
        return _Act()

    if name == "staircase":
        scale = float(p.get("scale", 1.0)); K = int(p.get("K", 7))
        class _Act(nn.Module):
            def forward(self, s): return StaircaseSTE.apply(s, scale, K)
        return _Act()

    if name == "hardtanh_q":
        K = int(p.get("K", 7))
        class _Act(nn.Module):
            def forward(self, s): return HardtanhQSTE.apply(s, K)
        return _Act()

    if name == "sigmoid_q":
        scale = float(p.get("scale", 4.0)); K = int(p.get("K", 7))
        class _Act(nn.Module):
            def forward(self, s): return SigmoidQSTE.apply(s, scale, K)
        return _Act()

    if name == "relu_q":
        scale = float(p.get("scale", 1.0)); K = int(p.get("K", 7))
        class _Act(nn.Module):
            def forward(self, s): return ReluQSTE.apply(s, scale, K)
        return _Act()

    if name == "square_q":
        scale = float(p.get("scale", 8.0)); K = int(p.get("K", 7))
        class _Act(nn.Module):
            def forward(self, s): return SquareQSTE.apply(s, scale, K)
        return _Act()

    raise ValueError(f"Unknown activation: {name!r}")


# Quick smoke test of the factory.
_dummy = make_activation(CONFIG["activation"], CONFIG["act_params"])
_s = torch.tensor([-3.0, -1.0, -0.1, 0.0, 0.1, 1.0, 3.0])
print(f"{CONFIG['activation']}({_s.tolist()}) = {_dummy(_s).tolist()}")

## 6. Model definition (`QuantMLP`)

A general FC stack `topology[0] → topology[1] → … → topology[-1]` with **integer-only** weights/biases derived on every forward pass via STEs:

* `TernarizeSTE` — used for layers tagged `kind="ternary"`. Forward: `sign(W) ⊙ 1[|W| ≥ thresh · mean(|W|)]` (Ternary Weight Networks, Li & Liu 2016). Backward: identity.
* `IntQuantSTE`  — used for layers tagged `kind="int"`. Forward: scales `W` so its largest magnitude maps to `max_int`, rounds, clamps to `[-max_int, +max_int]`. Backward: identity.
* `RoundSTE`     — used for every bias. Forward: `round(b)`. Backward: identity.

Continuous *shadow* weights are kept inside each `nn.Linear`; the optimizer (Adam) updates those, while the forward pass always projects them through the STE quantizers. Because the **forward pass already operates on integers**, the training-time accuracy is exactly the accuracy you'll see when the exported CSVs are loaded back — **no PTQ cliff**.

In [ ]:
class TernarizeSTE(torch.autograd.Function):
    """Forward: w -> sign(w) * 1[|w| >= thresh * mean(|w|)]. Backward: identity."""

    @staticmethod
    def forward(ctx, w, thresh_factor):
        thresh = float(thresh_factor) * w.abs().mean()
        return torch.sign(w) * (w.abs() >= thresh).to(w.dtype)

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output, None


def ternarize(w: torch.Tensor, thresh_factor: float = 0.7) -> torch.Tensor:
    return TernarizeSTE.apply(w, thresh_factor)


class IntQuantSTE(torch.autograd.Function):
    """Forward: round(w * (max_int / max(|w|))) clipped to ±max_int. Backward: identity."""

    @staticmethod
    def forward(ctx, w, max_int):
        w_max = w.abs().max().clamp(min=1e-8)
        scale = float(max_int) / w_max
        return torch.round(w * scale).clamp(-float(max_int), float(max_int))

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output, None


def int_quant(w: torch.Tensor, max_int: int = 15) -> torch.Tensor:
    return IntQuantSTE.apply(w, max_int)


class RoundSTE(torch.autograd.Function):
    """Forward: round(x). Backward: identity. Used for biases."""

    @staticmethod
    def forward(ctx, x):
        return torch.round(x)

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output


def round_ste(x: torch.Tensor) -> torch.Tensor:
    return RoundSTE.apply(x)


def _quantize_weight(w: torch.Tensor, spec: dict) -> torch.Tensor:
    kind = spec["kind"].lower()
    if kind == "ternary":
        return ternarize(w, spec.get("thresh", 0.7))
    if kind == "int":
        return int_quant(w, spec["max_int"])
    raise ValueError(f"Unknown layer_quant kind: {kind!r}  (use 'ternary' or 'int')")


class QuantMLP(nn.Module):
    """Parametric integer-quantized FC stack matching the C++ FHE layer format.

    For an N-Linear topology `[d0, d1, ..., dN]`, this stacks N `nn.Linear`
    layers. Each forward pass projects each layer's continuous shadow weight
    through the STE specified by `layer_quant[i]`, and applies
    `activation_module` after every Linear *except* the last one.
    """

    def __init__(self, topology: list[int], activation_name: str,
                 activation_params: dict, layer_quant: list[dict]):
        super().__init__()
        assert len(topology) >= 2, "topology must have at least 2 dims"
        assert len(layer_quant) == len(topology) - 1, \
            "layer_quant must have one entry per Linear layer"

        self.linears = nn.ModuleList(
            nn.Linear(topology[i], topology[i + 1]) for i in range(len(topology) - 1)
        )
        # One activation module per *hidden* layer (i.e. all Linears except the last).
        self.activations = nn.ModuleList(
            make_activation(activation_name, activation_params)
            for _ in range(len(topology) - 2)
        )
        self.layer_quant = layer_quant

    def quantized_weights(self):
        """Return parallel lists of integer (W_i, b_i) actually used by forward()."""
        Ws, bs = [], []
        for layer, spec in zip(self.linears, self.layer_quant):
            Ws.append(_quantize_weight(layer.weight, spec))
            bs.append(round_ste(layer.bias))
        return Ws, bs

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        Ws, bs = self.quantized_weights()
        for i, (W, b) in enumerate(zip(Ws, bs)):
            x = F.linear(x, W, b)
            if i < len(self.activations):
                x = self.activations[i](x)
        return x

    def hidden_preactivations(self, x: torch.Tensor) -> list[torch.Tensor]:
        """Return the integer pre-activations `s_i = W_i @ h_{i-1} + b_i` for ALL layers
        (including the output layer). Used by the headroom check in § 11."""
        Ws, bs = self.quantized_weights()
        h = x
        pre = []
        for i, (W, b) in enumerate(zip(Ws, bs)):
            s = F.linear(h, W, b)
            pre.append(s)
            if i < len(self.activations):
                h = self.activations[i](s)
        return pre


model = QuantMLP(
    topology=CONFIG["topology"],
    activation_name=CONFIG["activation"],
    activation_params=CONFIG["act_params"],
    layer_quant=CONFIG["layer_quant"],
).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print("Trainable params:", n_params)
for i, spec in enumerate(CONFIG["layer_quant"], start=1):
    print(f"  Linear {i} ({CONFIG['topology'][i-1]} -> {CONFIG['topology'][i]}): {spec}")

## 7. Training loop

In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"]
)
criterion = nn.CrossEntropyLoss()


def evaluate(model, loader, device):
    model.eval()
    correct = 0
    total = 0
    loss_sum = 0.0
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            loss_sum += loss.item() * xb.size(0)
            pred = logits.argmax(dim=1)
            correct += (pred == yb).sum().item()
            total += xb.size(0)
    return loss_sum / total, correct / total


history = {
    "epoch":      [],
    "train_loss": [],
    "train_acc":  [],
    "test_loss":  [],
    "test_acc":   [],
}

for epoch in range(1, CONFIG["epochs"] + 1):
    model.train()
    epoch_loss = 0.0
    epoch_correct = 0
    epoch_n = 0
    for xb, yb in train_loader:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        epoch_loss    += loss.item() * xb.size(0)
        epoch_correct += (logits.argmax(dim=1) == yb).sum().item()
        epoch_n       += xb.size(0)

    train_loss = epoch_loss / epoch_n
    train_acc  = epoch_correct / epoch_n
    test_loss, test_acc = evaluate(model, test_loader, DEVICE)

    history["epoch"].append(epoch)
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["test_loss"].append(test_loss)
    history["test_acc"].append(test_acc)

    print(
        f"epoch {epoch:3d}/{CONFIG['epochs']}  "
        f"train_loss={train_loss:.4f}  train_acc={train_acc*100:.2f}%  "
        f"test_loss={test_loss:.4f}  test_acc={test_acc*100:.2f}%"
    )

## 7b. Training curves

Two side-by-side plots from the per-epoch `history` collected above:

* **Loss vs epoch** — train loss (running average across the epoch's mini-batches) and test loss.
* **Accuracy vs epoch** — train accuracy (computed on the live mid-training predictions) and test accuracy.

In [ ]:
import matplotlib.pyplot as plt

epochs = history["epoch"]

fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(12, 4.5))

ax_loss.plot(epochs, history["train_loss"], label="train loss", marker="o", markersize=3)
ax_loss.plot(epochs, history["test_loss"],  label="test loss",  marker="o", markersize=3)
ax_loss.set_xlabel("epoch")
ax_loss.set_ylabel("cross-entropy loss")
ax_loss.set_title(f"CIFAR10 {' -> '.join(map(str, CONFIG['topology']))}: loss vs epoch")
ax_loss.grid(True, alpha=0.3)
ax_loss.legend()

ax_acc.plot(epochs, [a * 100 for a in history["train_acc"]], label="train acc", marker="o", markersize=3)
ax_acc.plot(epochs, [a * 100 for a in history["test_acc"]],  label="test acc",  marker="o", markersize=3)
ax_acc.set_xlabel("epoch")
ax_acc.set_ylabel("accuracy (%)")
ax_acc.set_title(f"CIFAR10 {' -> '.join(map(str, CONFIG['topology']))}: accuracy vs epoch")
ax_acc.grid(True, alpha=0.3)
ax_acc.legend()

best_epoch = int(np.argmax(history["test_acc"])) + 1
best_acc   = max(history["test_acc"]) * 100
print(f"Best test accuracy: {best_acc:.2f}% at epoch {best_epoch} (chance = 10.00%).")

fig.tight_layout()
plt.show()

## 8. Final evaluation

In [ ]:
final_train_loss, final_train_acc = evaluate(model, train_loader, DEVICE)
final_test_loss,  final_test_acc  = evaluate(model, test_loader,  DEVICE)
print(f"Final train accuracy: {final_train_acc*100:.2f}%  (loss {final_train_loss:.4f})")
print(f"Final test  accuracy: {final_test_acc*100:.2f}%  (loss {final_test_loss:.4f})")

## 9. FHE-side headroom validation

The C++ side encodes integer pre-activations into a CKKS plaintext modulo `pInput` (for hidden layers) or decodes them modulo `pOutput` (for the output layer). For the trainer's STE forward to match the deployed LUT bit-for-bit we need:

* **Hidden layers (LUT activation)** — `max |s_hidden| < preShift` (default `256`). If exceeded, the deployed LUT will *wrap* and produce wrong activation values on those samples — accuracy drops, but nothing crashes.
* **Output layer (no activation, modular decryption)** — `max |s_output| < pOutput / 2` (default `512`). Both the **empirical** maximum (computed on the test set) and a **worst-case bound** `w_max_int * fan_in + |b|_max` are reported. The worst case is often very loose for wide networks (e.g. 1024-hidden CIFAR-100 has worst case 15360 but practical pre-activations concentrate near 0); a soft warning is emitted only when the empirical max blows the budget.

In [ ]:
model.eval()

# Collect integer pre-activations for every layer over the entire test set.
per_layer_max = [0] * (len(CONFIG["topology"]) - 1)
with torch.no_grad():
    for xb, _ in test_loader:
        xb = xb.to(DEVICE)
        for i, s in enumerate(model.hidden_preactivations(xb)):
            per_layer_max[i] = max(per_layer_max[i], int(s.abs().max().item()))

# Worst-case bound for each layer: max|W| * fan_in + max|b|.
with torch.no_grad():
    Ws_t, bs_t = model.quantized_weights()
worst_case = [
    int(W.abs().max().item()) * W.shape[1] + int(b.abs().max().item())
    for W, b in zip(Ws_t, bs_t)
]

preShift   = int(CONFIG["preShift"])
out_budget = int(CONFIG["pOutput"]) // 2
nL = len(per_layer_max)

print(f"Hidden-layer pre-activation budget: |s| < preShift = {preShift}")
warned = False
for i in range(nL - 1):
    emp = per_layer_max[i]
    wc  = worst_case[i]
    status = "OK" if emp < preShift else "OVER"
    if emp >= preShift: warned = True
    print(f"  Linear {i+1} (hidden, fan_in={CONFIG['topology'][i]:>5d}): "
          f"empirical max|s| = {emp:>6d}   worst-case bound = {wc:>7d}   [{status}]")

print(f"\nOutput-layer modular-decryption budget: |s| < pOutput/2 = {out_budget}")
emp_out = per_layer_max[-1]
wc_out  = worst_case[-1]
status_emp = "OK" if emp_out < out_budget else "OVER (will wrap on some samples)"
status_wc  = "OK" if wc_out  < out_budget else "loose worst-case (empirical may still be fine)"
if emp_out >= out_budget: warned = True
print(f"  Linear {nL} (output, fan_in={CONFIG['topology'][-2]:>5d}): "
      f"empirical max|s| = {emp_out:>6d}   [{status_emp}]")
print(f"  worst-case bound = w_max * fan_in + |b|_max = {wc_out:>7d}   [{status_wc}]")

if warned:
    print("\n!! Soft warning: at least one empirical pre-activation exceeded the budget.")
    print("   The deployed LUT/decryption will wrap on those samples, hurting accuracy.")
    print("   Either widen preShift / pOutput in the C++ side, or shrink w_max / fan_in.")
else:
    print("\nAll pre-activations fit within the FHE-side budgets — safe to deploy.")

## 10. CSV export and run config

Two writes:

1. **Weights** — the **integer** weights actually used by the forward pass (no further quantization or rescaling), no headers, with `fmt="%d"`. This matches the file convention already used by `MNIST_30/`, `MNIST_100/`, `cifar10/`, etc.

   For an `N`-Linear topology, the export produces `2 * N` files:

   ```
   <output_dir>/<prefix>_W1.csv  <prefix>_b1.csv
   <output_dir>/<prefix>_W2.csv  <prefix>_b2.csv
   ...
   <output_dir>/<prefix>_WN.csv  <prefix>_bN.csv
   ```

   `torch.nn.Linear.weight` is already stored as `(out_features, in_features)`, which matches the C++ `LoadCsv2D(path, expected_in, expected_out)` preferred orientation, so we dump the parameters directly without transposing.

2. **Run config** — a single companion JSON file `<output_dir>/<prefix>_config.json` capturing everything needed to reproduce or audit this run: the network topology, hidden activation (name + params), per-layer quantization spec, FHE-side budgets (`preShift`, `pOutput`), training hyperparameters, final/best accuracies, the headroom report from § 9, and the list of weight files written above. The JSON is human-readable and small (a few KB).

In [ ]:
output_dir = Path(CONFIG["output_dir"])
output_dir.mkdir(parents=True, exist_ok=True)

model.eval()
with torch.no_grad():
    Ws_t, bs_t = model.quantized_weights()

Ws_int = [W.detach().cpu().numpy().astype(np.int64) for W in Ws_t]
bs_int = [b.detach().cpu().numpy().astype(np.int64) for b in bs_t]

# Sanity: every value really is an integer with the expected per-layer range.
for i, (W, spec) in enumerate(zip(Ws_int, CONFIG["layer_quant"]), start=1):
    if spec["kind"] == "ternary":
        assert set(np.unique(W).tolist()).issubset({-1, 0, 1}), \
            f"W{i} not ternary: unique={np.unique(W)}"
    elif spec["kind"] == "int":
        assert int(np.abs(W).max()) <= spec["max_int"], \
            f"W{i} outside +/- {spec['max_int']}: max(|W|)={int(np.abs(W).max())}"

prefix = CONFIG["prefix"]
paths: dict[str, Path] = {}
for i, (W, b) in enumerate(zip(Ws_int, bs_int), start=1):
    pW = output_dir / f"{prefix}_W{i}.csv"
    pb = output_dir / f"{prefix}_b{i}.csv"
    np.savetxt(pW, W, delimiter=",", fmt="%d")
    np.savetxt(pb, b, fmt="%d")
    paths[f"W{i}"] = pW
    paths[f"b{i}"] = pb
    print(f"  W{i}: shape={W.shape}  range=[{int(W.min())}, {int(W.max())}]   "
          f"unique={len(np.unique(W))}")
    print(f"  b{i}: shape={b.shape}  range=[{int(b.min())}, {int(b.max())}]")

print()
for name, p in paths.items():
    print(f"{name}: {p}  ({p.stat().st_size} bytes)")

# ── Write a companion JSON config alongside the CSV weights ────────────────
import datetime
import json as _json

config_path = output_dir / f"{prefix}_config.json"

run_config = {
    "dataset": "CIFAR10",
    "prefix":  prefix,
    "output_dir": str(output_dir),
    "model": {
        "topology":    list(CONFIG["topology"]),
        "activation":  CONFIG["activation"],
        "act_params":  dict(CONFIG["act_params"]),
        "layer_quant": [dict(q) for q in CONFIG["layer_quant"]],
        "n_params":    int(n_params),
    },
    "fhe": {
        "preShift": int(CONFIG["preShift"]),
        "pOutput":  int(CONFIG["pOutput"]),
    },
    "training": {
        "epochs":       int(CONFIG["epochs"]),
        "batch_size":   int(CONFIG["batch_size"]),
        "lr":           float(CONFIG["lr"]),
        "weight_decay": float(CONFIG["weight_decay"]),
        "seed":         int(CONFIG["seed"]),
        "device":       str(DEVICE),
    },
    "results": {
        "final_train_acc":     float(final_train_acc),
        "final_train_loss":    float(final_train_loss),
        "final_test_acc":      float(final_test_acc),
        "final_test_loss":     float(final_test_loss),
        "best_test_acc":       float(max(history["test_acc"])),
        "best_test_acc_epoch": int(history["epoch"][int(np.argmax(history["test_acc"]))]),
    },
    "headroom": {
        "preShift_budget":         int(CONFIG["preShift"]),
        "pOutput_half_budget":     int(CONFIG["pOutput"]) // 2,
        "empirical_max_abs_pre":   [int(v) for v in per_layer_max],
        "worst_case_pre_bound":    [int(v) for v in worst_case],
    },
    "weights": {
        "fmt":   "%d",
        "files": [p.name for p in paths.values()],
    },
    "torch_version": torch.__version__,
    "timestamp":     datetime.datetime.now().isoformat(timespec="seconds"),
}

with open(config_path, "w") as f:
    _json.dump(run_config, f, indent=2)

print(f"\nconfig: {config_path}  ({config_path.stat().st_size} bytes)")


## 11. Exact-round-trip reload check

Reload the CSVs with NumPy and check shapes **and exact integer parity** (since we wrote `%d`, the round-trip must be lossless: `max |W − W_reloaded| == 0`). Then re-run inference using the reloaded integers via a pure-NumPy path that mirrors the C++ side bit-for-bit, and assert **100 %** prediction agreement with the live PyTorch model.

In [ ]:
def _activation_numpy(name: str, params: dict):
    """Pure-NumPy mirror of make_activation, applied per-element to int64 arrays."""
    name = name.lower()
    p = dict(params or {})
    if name == "sign":
        return lambda s: np.where(s >= 0, 1, -1).astype(np.int64)
    if name == "heaviside":
        return lambda s: np.where(s >= 0, 1, 0).astype(np.int64)
    if name == "ternary_act":
        thr = float(p.get("thresh", 1.0))
        def f(s):
            out = np.zeros_like(s, dtype=np.int64)
            out = np.where(s >  thr,  1, out)
            out = np.where(s < -thr, -1, out)
            return out
        return f
    if name == "staircase":
        scale = float(p.get("scale", 1.0)); K = int(p.get("K", 7))
        return lambda s: np.clip(np.round(s / scale).astype(np.int64), -K, K)
    if name == "hardtanh_q":
        K = int(p.get("K", 7))
        return lambda s: np.round(np.clip(s, -K, K)).astype(np.int64)
    if name == "sigmoid_q":
        scale = float(p.get("scale", 4.0)); K = int(p.get("K", 7))
        return lambda s: np.clip(np.round(K / (1.0 + np.exp(-s / scale))).astype(np.int64), 0, K)
    if name == "relu_q":
        scale = float(p.get("scale", 1.0)); K = int(p.get("K", 7))
        return lambda s: np.clip(np.round(s / scale).astype(np.int64), 0, K)
    if name == "square_q":
        scale = float(p.get("scale", 8.0)); K = int(p.get("K", 7))
        return lambda s: np.clip(np.round((s / scale) ** 2).astype(np.int64), 0, K)
    raise ValueError(f"Unknown activation: {name!r}")


# Reload all weights and check integer parity.
Ws_rl: list[np.ndarray] = []
bs_rl: list[np.ndarray] = []
for i in range(1, len(Ws_int) + 1):
    W_rl = np.loadtxt(paths[f"W{i}"], delimiter=",", dtype=np.int64)
    b_rl = np.loadtxt(paths[f"b{i}"],                dtype=np.int64)
    # 1-D `np.savetxt` with shape (1,) yields a 0-D scalar on reload; reshape.
    if b_rl.ndim == 0:
        b_rl = b_rl.reshape(1)
    Ws_rl.append(W_rl)
    bs_rl.append(b_rl)

# Shape + exact integer round-trip checks.
for i, (W, b, W_rl, b_rl) in enumerate(zip(Ws_int, bs_int, Ws_rl, bs_rl), start=1):
    assert W_rl.shape == W.shape, f"W{i} shape {W_rl.shape} != {W.shape}"
    assert b_rl.shape == b.shape, f"b{i} shape {b_rl.shape} != {b.shape}"
    assert np.array_equal(W, W_rl), f"W{i} round-trip mismatch"
    assert np.array_equal(b, b_rl), f"b{i} round-trip mismatch"
    print(f"  Linear {i}: max |W{i} - W{i}_rl| = {int(np.max(np.abs(W - W_rl)))}, "
          f"max |b{i} - b{i}_rl| = {int(np.max(np.abs(b - b_rl)))}")

print("All shapes and integer values OK.")


def numpy_inference(X: np.ndarray, Ws: list[np.ndarray], bs: list[np.ndarray],
                    activation_name: str, activation_params: dict) -> np.ndarray:
    """Pure-integer forward pass through the reloaded CSVs, mirroring the C++ side."""
    act = _activation_numpy(activation_name, activation_params)
    h = X.astype(np.int64)
    for i, (W, b) in enumerate(zip(Ws, bs)):
        s = h @ W.astype(np.int64).T + b.astype(np.int64)
        if i < len(Ws) - 1:
            h = act(s)
        else:
            return np.argmax(s, axis=1)
    raise RuntimeError("unreachable")


csv_pred = numpy_inference(
    X_test.astype(np.int64), Ws_rl, bs_rl,
    CONFIG["activation"], CONFIG["act_params"],
)
csv_acc = float((csv_pred == y_test).mean())

model.eval()
torch_pred_chunks = []
with torch.no_grad():
    for xb, _ in test_loader:
        xb = xb.to(DEVICE)
        torch_pred_chunks.append(model(xb).argmax(dim=1).cpu().numpy())
torch_pred = np.concatenate(torch_pred_chunks)
torch_acc  = float((torch_pred == y_test).mean())

agreement = float((csv_pred == torch_pred).mean())

print(f"PyTorch test accuracy:    {torch_acc*100:.2f}%")
print(f"Reloaded CSV accuracy:    {csv_acc*100:.2f}%")
print(f"Prediction agreement:     {agreement*100:.2f}%   (expected: 100.00% — pure-integer round-trip)")
assert agreement == 1.0, "CSV reload disagrees with live model — round-trip is broken."

## 12. Summary — paste-ready C++ wiring

Prints a `Network` builder snippet assembled from the actual CONFIG (topology, activation, prefix, output_dir), so the printed C++ matches whatever variant you trained. Drop it into any sub-project's `main.cpp` (e.g. `cifar10/main.cpp`) and copy/symlink the CSVs from `new_models/<output_dir>/` next to the sub-project's `CMakeLists.txt`.

In [ ]:
def _cpp_activation(name: str, params: dict, preShift: int) -> str:
    """Render a `fhednn::activations::...` call appropriate for the trained activation."""
    name = name.lower()
    p = dict(params or {})
    if name == "sign":
        return f"activations::Sign(/*preShift=*/{preShift}, ctx.params().pInput)"
    if name == "heaviside":
        return (f"activations::Step(/*threshold=*/{preShift}, /*low=*/0, /*high=*/1, "
                "ctx.params().pInput)")
    if name == "ternary_act":
        thr = int(p.get("thresh", 1))
        return (f"activations::Custom(\"ternary_act\",\n"
                f"            [](std::int64_t s) -> std::int64_t {{\n"
                f"                std::int64_t x = s - {preShift};\n"
                f"                if (x >  {thr}) return  1;\n"
                f"                if (x < -{thr}) return -1;\n"
                f"                return 0;\n"
                f"            }},\n"
                f"            /*preShift=*/{preShift}, ctx.params().pInput)")
    if name == "staircase":
        scale = float(p.get("scale", 1.0)); K = int(p.get("K", 7))
        return (f"activations::Custom(\"staircase\",\n"
                f"            [](std::int64_t s) -> std::int64_t {{\n"
                f"                double x = static_cast<double>(s - {preShift}) / {scale};\n"
                f"                std::int64_t q = static_cast<std::int64_t>(std::llround(x));\n"
                f"                return std::max<std::int64_t>(-{K}, std::min<std::int64_t>({K}, q));\n"
                f"            }},\n"
                f"            /*preShift=*/{preShift}, ctx.params().pInput)")
    return (f"activations::Custom(\"{name}\",\n"
            f"            /* TODO: port the python forward to C++; mirror make_activation in train_mnist.ipynb */,\n"
            f"            /*preShift=*/{preShift}, ctx.params().pInput)")


topology = CONFIG["topology"]
nL       = len(topology) - 1
prefix   = CONFIG["prefix"]
outdir   = CONFIG["output_dir"]
preShift = int(CONFIG["preShift"])
hidden_act = _cpp_activation(CONFIG["activation"], CONFIG["act_params"], preShift)

lines = []
lines.append("// ── Paste into <subproject>/main.cpp ──────────────────────────────────")
lines.append(f"// Trained from new_models/notebooks/train_cifar10.ipynb")
lines.append(f"// Topology   : {' -> '.join(map(str, topology))}")
lines.append(f"// Activation : {CONFIG['activation']} {CONFIG['act_params']}")
lines.append(f"// Output dir : new_models/{outdir}/   (copy/symlink {prefix}_W*.csv + _b*.csv next to CMakeLists.txt)")
lines.append("")
for i, d in enumerate(topology):
    name = "IN_DIM" if i == 0 else ("OUT_DIM" if i == nL else f"H{i}_DIM")
    lines.append(f"static constexpr int {name} = {d};")
lines.append("")
for i in range(1, nL + 1):
    in_name  = "IN_DIM" if i == 1   else (f"H{i-1}_DIM")
    out_name = "OUT_DIM" if i == nL else (f"H{i}_DIM")
    lines.append(f"auto W{i} = io::LoadCsv2D(\"../{prefix}_W{i}.csv\", {in_name}, {out_name});")
    lines.append(f"auto b{i} = io::LoadCsv1D(\"../{prefix}_b{i}.csv\");")
lines.append("")
lines.append("auto pixels = io::LoadImageBipolar(argv[1], IN_DIM, /*channels=*/3);")
lines.append("")
lines.append("FHEContext ctx;")
lines.append("Network    net;")
lines.append("net.SetInputShift(1)")
for i in range(1, nL + 1):
    lines.append(f"   .Linear(W{i}, b{i})" + ("" if i == nL else ""))
    if i < nL:
        lines.append(f"   .Activate({hidden_act})")
lines.append("   ;")
lines.append("net.Compile(ctx);")
lines.append("auto scores = net.Run(pixels);")
lines.append("// ──────────────────────────────────────────────────────────────────────")

print("\n".join(lines))